# Validation: quantifying error in detection, calibration, and clustering

*Using Projective Transformation for the Spatial Analysis of Team Behaviors in Football*

The other notebooks (`00`-`03`) take the pipeline's output at face value. This one instead
measures how much to trust it, for the thesis's validation/methodology section.

**Runs automatically, top to bottom, with no manual input required** - same as `00`-`03` - and
now over **every clip in `Goals/`, `BuildingAction/`, and `SetPieces/`**, not just one. Results
accumulate into one CSV across runs (`validation_all_clips.csv` - re-running this notebook, or
running it again after adding new clips, merges in rather than overwriting what's already there).

Two layers:

1. **Automatic checks (always on, every clip).** No ground truth exists for this project's own
   footage, so these can't report "true" accuracy - but they need zero manual work and still catch
   real problems: is the pitch calibration's error small when tested on points it wasn't fit from,
   are per-class detection counts/confidences stable across the clip, are the two team colour
   clusters actually well-separated. One row per clip, for every clip found.
2. **Optional manual sections (off by default, like `USE_MANUAL_FALLBACK` in notebook `00`), for
   ONE clip at a time.** Flip a `RUN_MANUAL_...` flag to `True` only if you want a real, precise
   number (precision/recall against hand-labeled boxes, clustering accuracy against hand-labeled
   true team) for the thesis - this needs a few minutes of hand-labeling per clip you check this
   way. Leave every flag `False` and the notebook is fully automatic end to end.

## 1. Clone the repository and install dependencies

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Batomet/Magisterka.git"
BRANCH = "claude/field-position-detection-dqda1g"
REPO_DIR = "/content/Magisterka"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log -1 --oneline

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR + "/src")

In [ ]:
!pip install -q -r requirements.txt

## 2. Mount Google Drive and list every clip

In [ ]:
from pitchvision import DriveConfig, VideoFrames, list_videos, mount_drive

mount_drive()

# Adjust `root` if BuildingAction/Goals/SetPieces don't live directly under My Drive.
drive_cfg = DriveConfig(root="/content/drive/MyDrive/Magisterka")

goal_videos = list_videos(drive_cfg.goals_path)
buildup_videos = list_videos(drive_cfg.building_action_path)
set_piece_videos = list_videos(drive_cfg.set_pieces_path)

all_clips = (
    [(p, "Goals") for p in goal_videos]
    + [(p, "BuildingAction") for p in buildup_videos]
    + [(p, "SetPieces") for p in set_piece_videos]
)
print(f"Goals: {len(goal_videos)}, BuildingAction: {len(buildup_videos)}, SetPieces: {len(set_piece_videos)}")
print(f"Total: {len(all_clips)} clips")

# Used later, by the OPTIONAL manual sections only (those work on one clip at a time).
sample_video = goal_videos[0]  # change to inspect a different clip manually
frames = VideoFrames(sample_video)
print("\nManual-section clip:", sample_video, "-", frames.frame_count, "frames @", frames.fps, "fps")

## 3. Shared model setup (downloaded once)

Both models here only ever do single-image inference (`.detect()`/`.calibrate()`), not
`model.track(..., persist=True)` - so, unlike `PlayerTracker` in `01`/`02`/`03`, there's no
tracker-state-leaking-across-clips risk in reusing the SAME instance for every clip below.

In [ ]:
from pitchvision import (
    PitchKeypointDetector,
    PlayerBallDetector,
    SPORTS_DETECTION_CLASSES,
    download_pitch_keypoint_weights,
    download_player_detection_weights,
)

pitch_weights_path = download_pitch_keypoint_weights(
    "/content/drive/MyDrive/pitchvision_models/football-pitch-detection.pt"
)
keypoint_detector = PitchKeypointDetector(weights=pitch_weights_path, confidence=0.5)

player_weights_path = download_player_detection_weights(
    "/content/drive/MyDrive/pitchvision_models/football-player-detection.pt"
)
detector = PlayerBallDetector(
    weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
)
SPECIALIZED_CLASS_NAMES = ("player", "goalkeeper", "referee", "ball")
print("Shared weights ready.")

## 4. Automatic validation across every clip

No ground truth exists for this project's own footage, so none of this is "true" accuracy - it's
three zero-labeling proxy checks, computed the same way for every clip:

- **Calibration**: held-out reprojection error (see notebook `00`'s calibration section) computed
  from `PitchKeypointDetector`'s own automatically-detected keypoints - their true pitch position
  is a known Laws-of-the-Game constant, not something a human needs to label.
- **Detection**: per-class count/confidence stability across sampled frames - wild swings usually
  mean missed detections (occlusion/motion blur), not an actual change in what's visible.
- **Clustering**: silhouette score for the two jersey-colour clusters - near `1` = well separated,
  near `0`/negative = the clustering probably isn't finding real team structure.

A clip that fails outright (e.g. video won't open) is skipped with its error recorded, not silently
dropped.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from pitchvision import (
    TeamClassifier,
    collect_jersey_colors,
    compute_calibration_holdout_error,
    pitch_keypoint_template,
)

MIN_POINTS_FOR_HOLDOUT = 8  # enough margin to both fit (>=4) and hold out (>=1) meaningfully
DETECTION_CHECK_STRIDE = 10
DETECTION_CHECK_MAX_FRAMES = 120


def automatic_validation_for_clip(video_path, source_folder):
    clip_frames = VideoFrames(video_path)
    clip_name = os.path.splitext(os.path.basename(video_path))[0]
    result = {"clip": clip_name, "source_folder": source_folder, "n_frames": clip_frames.frame_count}

    # --- calibration: held-out reprojection error from automatically-detected keypoints ---
    first_frame = clip_frames.read_frame(0)
    xy, conf = keypoint_detector.detect(first_frame)
    template = pitch_keypoint_template()
    confident = conf >= keypoint_detector.confidence
    result["n_confident_keypoints"] = int(confident.sum())
    if confident.sum() >= MIN_POINTS_FOR_HOLDOUT:
        holdout = compute_calibration_holdout_error(xy[confident], template[confident], n_repeats=30)
        result["calibration_mean_error_m"] = holdout["mean_error_m"]
        result["calibration_std_error_m"] = holdout["std_error_m"]
        result["calibration_max_error_m"] = holdout["max_error_m"]
    else:
        result["calibration_mean_error_m"] = float("nan")
        result["calibration_std_error_m"] = float("nan")
        result["calibration_max_error_m"] = float("nan")

    # --- detection: per-class count/confidence stability across sampled frames ---
    check_frame_indices = list(range(0, min(clip_frames.frame_count, DETECTION_CHECK_MAX_FRAMES), DETECTION_CHECK_STRIDE))
    count_rows, all_confidences = [], []
    for i in check_frame_indices:
        dets = detector.detect(clip_frames.read_frame(i))
        counts = {cls: 0 for cls in SPECIALIZED_CLASS_NAMES}
        for d in dets:
            counts[d.class_name] = counts.get(d.class_name, 0) + 1
            all_confidences.append(d.confidence)
        count_rows.append(counts)
    counts_df = pd.DataFrame(count_rows, columns=list(SPECIALIZED_CLASS_NAMES))
    for cls in SPECIALIZED_CLASS_NAMES:
        result[f"{cls}_count_mean"] = counts_df[cls].mean()
        result[f"{cls}_count_std"] = counts_df[cls].std()
    result["mean_detection_confidence"] = float(np.mean(all_confidences)) if all_confidences else float("nan")

    # --- clustering: silhouette score for the jersey-colour split ---
    jersey_colors = collect_jersey_colors(video_path, detector, class_names=("player",), stride=30)
    result["jersey_n_samples"] = len(jersey_colors)
    silhouette = float("nan")
    if len(jersey_colors) >= 2:
        team_classifier = TeamClassifier(n_clusters=2).fit(jersey_colors)
        cluster_labels = team_classifier.predict_from_colors(jersey_colors)
        if len(set(cluster_labels)) > 1:
            scaled = StandardScaler().fit_transform(jersey_colors)
            silhouette = silhouette_score(scaled, cluster_labels)
    result["jersey_cluster_silhouette"] = silhouette

    return result

In [ ]:
validation_rows = []
failed = []
for video_path, source_folder in all_clips:
    clip_name = os.path.splitext(os.path.basename(video_path))[0]
    print(f"Validating {source_folder}/{clip_name}...")
    try:
        validation_rows.append(automatic_validation_for_clip(video_path, source_folder))
    except Exception as e:
        print(f"  SKIPPED ({type(e).__name__}: {e})")
        failed.append({"clip": clip_name, "source_folder": source_folder, "error": str(e)})

validation_df = pd.DataFrame(validation_rows)
print(f"\nValidated {len(validation_rows)}/{len(all_clips)} clips successfully.")
if failed:
    print("Skipped clips:", failed)
validation_df

**Save all clips, not just this run's.** Re-running this notebook (or running it again after new
clips are added) merges the new results into `validation_all_clips.csv` rather than overwriting it
- a clip that's re-validated replaces its own old row (by `clip` + `source_folder`), everything else
already in the file is kept.

In [ ]:
output_dir = "/content/drive/MyDrive/pitchvision_outputs"
os.makedirs(output_dir, exist_ok=True)
validation_csv_path = os.path.join(output_dir, "validation_all_clips.csv")

if os.path.exists(validation_csv_path):
    existing_df = pd.read_csv(validation_csv_path)
    combined_df = pd.concat([existing_df, validation_df], ignore_index=True)
    combined_df = combined_df.drop_duplicates(subset=["clip", "source_folder"], keep="last")
else:
    combined_df = validation_df

combined_df = combined_df.sort_values(["source_folder", "clip"]).reset_index(drop=True)
combined_df.to_csv(validation_csv_path, index=False)
print(f"Saved {len(combined_df)} total clip validation rows (across all runs) to {validation_csv_path}")
combined_df

### Aggregate view across all validated clips

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(combined_df["calibration_mean_error_m"].dropna(), bins=15)
axes[0].set_xlabel("Mean holdout calibration error (m)")
axes[0].set_ylabel("Clips")
axes[0].set_title("Calibration error across all clips")

axes[1].hist(combined_df["jersey_cluster_silhouette"].dropna(), bins=15)
axes[1].set_xlabel("Silhouette score")
axes[1].set_title("Team-colour cluster separation across all clips")

by_folder = combined_df.groupby("source_folder")["mean_detection_confidence"].mean()
axes[2].bar(by_folder.index, by_folder.values)
axes[2].set_ylabel("Mean detection confidence")
axes[2].set_title("Detection confidence by folder")

plt.tight_layout()
plt.show()

print(combined_df.groupby("source_folder")[
    ["calibration_mean_error_m", "jersey_cluster_silhouette", "mean_detection_confidence"]
].mean())

## 5. Optional manual sections (real numbers, one clip at a time)

Everything below works on a single clip (`sample_video`, set in section 2) rather than the whole
batch - hand-labeling doesn't scale to every clip, so these are for spot-checking a precise number
on a clip or two, not for the full dataset. Every flag defaults to `False`; leave them all off and
these cells are a no-op.

### Optional: manual calibration fallback

Off by default - only turn this on if section 4's automatic detection couldn't find enough
confident keypoints for `sample_video` (heavy occlusion, an unusual crop) and you want a working
calibrator for it anyway. Same hover-to-read-pixel-coordinates pattern as notebook `00`'s manual
fallback: hand-pick 10+ landmarks (more than the minimum 4 a fit needs) so a holdout split is still
possible, fill them into `landmark_pixels`.

In [ ]:
import cv2
import plotly.express as px

from pitchvision import PITCH_LANDMARKS_M, PitchCalibrator

# Set this to True only if automatic detection couldn't find enough confident keypoints for
# `sample_video`. With this False (the default), this cell is a no-op.
USE_MANUAL_CALIBRATION_FALLBACK = False

if not USE_MANUAL_CALIBRATION_FALLBACK:
    print("Skipping manual calibration fallback (USE_MANUAL_CALIBRATION_FALLBACK is False).")
else:
    first_frame = frames.read_frame(0)
    fig = px.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
    fig.update_layout(title="Hover to read pixel coordinates for the landmarks below", height=700)
    fig.show()
    print("Available landmark names:", list(PITCH_LANDMARKS_M.keys()))

    # EDIT THIS: at least 10 landmarks, read off the hover tooltip above. Do not leave these
    # defaults - they are placeholders and do not correspond to real points in your video.
    landmark_pixels = {
        "top_left_corner": (50, 60),
        "top_right_corner": (1200, 55),
        "bottom_left_corner": (10, 650),
        "bottom_right_corner": (1250, 640),
        "centre_spot": (630, 340),
        "centre_top": (630, 55),
        "centre_bottom": (630, 650),
        "left_penalty_top": (150, 200),
        "left_penalty_bottom": (150, 480),
        "left_penalty_spot": (220, 340),
        "left_six_yard_top": (80, 260),
        "left_six_yard_bottom": (80, 420),
    }

    _PLACEHOLDER = {
        "top_left_corner": (50, 60),
        "top_right_corner": (1200, 55),
        "bottom_left_corner": (10, 650),
        "bottom_right_corner": (1250, 640),
        "centre_spot": (630, 340),
        "centre_top": (630, 55),
        "centre_bottom": (630, 650),
        "left_penalty_top": (150, 200),
        "left_penalty_bottom": (150, 480),
        "left_penalty_spot": (220, 340),
        "left_six_yard_top": (80, 260),
        "left_six_yard_bottom": (80, 420),
    }
    assert landmark_pixels != _PLACEHOLDER, (
        "landmark_pixels still holds the placeholder values - edit them with real pixel "
        "coordinates read off the hover tooltip above before continuing."
    )

    pixel_points = np.array(list(landmark_pixels.values()))
    pitch_points = np.array([PITCH_LANDMARKS_M[name] for name in landmark_pixels])

    manual_holdout_result = compute_calibration_holdout_error(pixel_points, pitch_points, n_repeats=30)
    print(f"mean holdout error: {manual_holdout_result['mean_error_m']:.3f} m")

    manual_calibrator = PitchCalibrator.from_point_pairs(pixel_points, pitch_points)

### Optional: manual detection labeling (real precision/recall/F1)

Off by default. Pick a few frames of `sample_video`, hover to read pixel coordinates for the boxes
actually visible, and compare against the detector's predictions on those same frames with
IoU-matched precision/recall/F1/mean-IoU per class.

In [ ]:
# Set this to True to hand-label a few frames for a real precision/recall number. With this False
# (the default), this section is a no-op.
RUN_MANUAL_DETECTION_LABELING = False

if RUN_MANUAL_DETECTION_LABELING:
    EVAL_FRAME_INDICES = sorted({0, frames.frame_count // 3, 2 * frames.frame_count // 3})
    eval_frames = {i: frames.read_frame(i) for i in EVAL_FRAME_INDICES}
    frame_detections = {i: detector.detect(frame) for i, frame in eval_frames.items()}
    for i, dets in frame_detections.items():
        print(f"frame {i}: {len(dets)} detections")
else:
    print("Skipping manual detection labeling (RUN_MANUAL_DETECTION_LABELING is False).")

In [ ]:
if RUN_MANUAL_DETECTION_LABELING:
    for i, frame in eval_frames.items():
        fig = px.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        fig.update_layout(title=f"Frame {i} - hover to read box corner pixel coordinates", height=700)
        fig.show()
else:
    print("Skipping (RUN_MANUAL_DETECTION_LABELING is False).")

In [ ]:
from pitchvision import compute_detection_metrics, ground_truth_boxes_dataframe, predicted_boxes_dataframe

if RUN_MANUAL_DETECTION_LABELING:
    # EDIT THIS: for each frame index above, list every (class_name, x1, y1, x2, y2) box you can
    # confidently place by eye. class_name must be one of "player", "goalkeeper", "referee", "ball".
    # Do not leave these defaults - they are placeholders and do not correspond to real boxes.
    ground_truth_boxes = {
        EVAL_FRAME_INDICES[0]: [
            ("player", 100, 200, 140, 320),
            ("player", 300, 180, 340, 300),
            ("ball", 500, 400, 515, 415),
        ],
    }

    _PLACEHOLDER = {
        EVAL_FRAME_INDICES[0]: [
            ("player", 100, 200, 140, 320),
            ("player", 300, 180, 340, 300),
            ("ball", 500, 400, 515, 415),
        ],
    }
    assert ground_truth_boxes != _PLACEHOLDER, (
        "ground_truth_boxes still holds the placeholder values - edit them with real boxes read "
        "off the hover tooltips above before continuing."
    )

    predicted_df = predicted_boxes_dataframe(frame_detections)
    ground_truth_df = ground_truth_boxes_dataframe(ground_truth_boxes)
    detection_metrics_df = compute_detection_metrics(predicted_df, ground_truth_df, iou_threshold=0.5)
else:
    detection_metrics_df = None
    print("Skipping (RUN_MANUAL_DETECTION_LABELING is False).")

detection_metrics_df

### Optional: manual clustering accuracy (real accuracy against hand-labeled tracks)

Off by default. Runs a short tracked window of `sample_video`, shows a grid of tracks' crops, and
lets you hand-label true team per track - compared against the resolved `team_id` under the optimal
cluster-id-to-team matching.

In [ ]:
from pitchvision import PlayerTracker, TrackingPipeline

# Set this to True to hand-label a sample of tracks for a real clustering-accuracy number. With
# this False (the default), this section is a no-op.
RUN_MANUAL_CLUSTER_LABELING = False

if RUN_MANUAL_CLUSTER_LABELING:
    manual_jersey_colors = collect_jersey_colors(sample_video, detector, class_names=("player",), stride=30)
    manual_team_classifier = TeamClassifier(n_clusters=2).fit(manual_jersey_colors)
    print("cluster_swatches (RGB):", manual_team_classifier.cluster_swatches)

    manual_calibrator_for_tracking = manual_calibrator if USE_MANUAL_CALIBRATION_FALLBACK else keypoint_detector.calibrate(frames.read_frame(0))

    tracker = PlayerTracker(
        weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
    )
    pipeline = TrackingPipeline(
        tracker=tracker,
        calibrator=manual_calibrator_for_tracking,
        team_classifier=manual_team_classifier,
        team_eligible_class_names=("player",),
    )
    eval_tracks_df = pipeline.run(sample_video, max_frames=90)
    print(eval_tracks_df["class_name"].value_counts())
else:
    print("Skipping manual clustering labeling (RUN_MANUAL_CLUSTER_LABELING is False).")

In [ ]:
if RUN_MANUAL_CLUSTER_LABELING:
    N_TRACKS_TO_LABEL = 16

    player_rows = eval_tracks_df[eval_tracks_df["class_name"] == "player"]
    first_seen = player_rows.sort_values("frame").drop_duplicates("track_id")
    sample_tracks = first_seen.head(N_TRACKS_TO_LABEL)

    crops, track_ids = [], []
    for _, row in sample_tracks.iterrows():
        frame = frames.read_frame(int(row["frame"]))
        x1, y1, x2, y2 = int(row["bbox_x1"]), int(row["bbox_y1"]), int(row["bbox_x2"]), int(row["bbox_y2"])
        crops.append(frame[max(y1, 0):y2, max(x1, 0):x2])
        track_ids.append(int(row["track_id"]))

    n_cols = 4
    n_rows = (len(crops) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))
    axes = np.atleast_2d(axes)
    for idx in range(n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col]
        ax.set_xticks([])
        ax.set_yticks([])
        if idx >= len(crops):
            ax.axis("off")
            continue
        ax.imshow(cv2.cvtColor(crops[idx], cv2.COLOR_BGR2RGB))
        ax.set_title(f"track {track_ids[idx]}", fontsize=9)
    plt.tight_layout()
    plt.show()
    print("track_ids shown, in order:", track_ids)
else:
    print("Skipping (RUN_MANUAL_CLUSTER_LABELING is False).")

In [ ]:
from pitchvision import compute_clustering_accuracy

if RUN_MANUAL_CLUSTER_LABELING:
    # EDIT THIS: for as many of the track_ids printed above as you reasonably can (at least a
    # handful), the TRUE team by eye ("A" or "B" - which physical team, not a cluster id). Do not
    # leave these defaults - they are placeholders.
    true_team_by_track = {
        track_ids[0]: "A",
        track_ids[1]: "A",
        track_ids[2]: "B",
        track_ids[3]: "B",
    }

    _PLACEHOLDER = {
        track_ids[0]: "A",
        track_ids[1]: "A",
        track_ids[2]: "B",
        track_ids[3]: "B",
    }
    assert true_team_by_track != _PLACEHOLDER, (
        "true_team_by_track still holds the placeholder values - label the tracks shown above "
        "(or at least a handful of them) before continuing."
    )

    labeled_tracks = eval_tracks_df[eval_tracks_df["track_id"].isin(true_team_by_track)].drop_duplicates("track_id")
    true_labels = labeled_tracks["track_id"].map(true_team_by_track)
    predicted_labels = labeled_tracks["team_id"]

    clustering_result = compute_clustering_accuracy(true_labels, predicted_labels)
    print(f"accuracy: {clustering_result['accuracy']:.3f} ({clustering_result['n_samples']} labeled tracks)")
    print("cluster id -> true team:", clustering_result["cluster_to_true_label"])
else:
    clustering_result = None
    print("Skipping (RUN_MANUAL_CLUSTER_LABELING is False).")

clustering_result["confusion_matrix"] if clustering_result is not None else None

## Summary

Always-on, zero-labeling checks, across every clip found:

- `validation_all_clips.csv` on Drive - one row per clip, accumulated across every time this
  notebook has been run (not just this run's clips). `combined_df` in this session holds the same
  data. Columns: `calibration_mean_error_m`/`std`/`max`, per-class detection count mean/std,
  `mean_detection_confidence`, `jersey_cluster_silhouette`, `jersey_n_samples`.

If you switched on any `RUN_MANUAL_.../USE_MANUAL_...` flag for `sample_video`:

- **Detection**: `detection_metrics_df` - precision/recall/F1/mean IoU, per class and overall.
- **Clustering**: `clustering_result["accuracy"]` - team-assignment accuracy against hand-labeled tracks.

These automatic numbers are proxies, not ground-truth accuracy - a clip with a high
`calibration_mean_error_m` or a low `jersey_cluster_silhouette` in `validation_all_clips.csv` is a
good candidate to check with the manual sections above before trusting its numbers elsewhere in
the thesis.